### 训练

在特征筛选测试给出的结果中选择10个候选特征作为输入来进行svm模型的训练

In [1]:
# -*- coding: utf-8 -*-
"""GAN-GRU 同款自动特征计算流程的 SVM 训练入口。

直接粘贴到 BigQuant Notebook 的一个单元运行。训练函数依据 FEATURE_SPEC
自动读取原始数据并计算 10 个基础因子；调用方不传 feature_panel 或 price_panel。
每次成功训练都会在 factor_lib/model_artifacts/svm_model_bundles 下创建独立模型包。
"""

from pathlib import Path
import importlib.util
import sys

import pandas as pd
from IPython.display import display


SVM_TRAINING_CONFIG = {
    "anchor_date": "2025-02-14",
    "training_start_date": "2017-01-03",
    "training_end_date": "2021-12-31",
    "validation_start_date": "2022-01-04",
    "validation_end_date": "2024-12-31",
    "universe": {"type": "all_a"},
    "signal_interval_trading_days": 20,
    "prediction_label_window_trading_days": 20,
    "label_config": {
        "return_definition": "universe_equal_weight_excess",
        "entry_price_field": "open",
        "entry_offset_trading_days": 1,
        "exit_price_field": "open",
        "positive_quantile": 0.30,
        "negative_quantile": 0.30,
        "min_cross_section_size": 30,
    },
    "preprocessing_config": {
        "winsorize_mad": True,
        "mad_limit": 5.0,
        "missing_policy": "cross_section_median",
        "neutralize_size_industry": False,
        "zscore": True,
        "global_standard_scaler": True,
    },
    "model_config": {
        "kernel": "rbf",  # rbf 即高斯核。
        "hyperparameter_search": {
            "C_values": [0.1, 0.3, 1.0, 3.0, 10.0],
            "gamma_values": [0.01, 0.03, 0.1, 0.3, 1.0],
            "metric": "rank_ic",  # 逐信号日横截面 RankIC 的时间均值。
        },
        "class_weight": "balanced",
        "cache_size": 600.0,
        "tol": 1e-3,
        "max_iter": -1,
    },
    "refit_on_train_and_validation": True,
    "progress_every": 5,
}

# 这是由本次初筛冻结的特征定义；顺序就是 SVM 输入维度顺序。
FEATURE_SPEC = [
    {"factor_name": "id2_std_nm", "feature_name": "id2_std_nm_3m", "params": {"market_index": "csi_all_share", "min_cs_count": 200, "min_industry_count": 5, "min_style_universe": 200, "min_ts_observations": 40, "n_months": 3, "neutralize_industry": True, "standardize_residual": False, "style_lower_quantile": 0.3, "style_upper_quantile": 0.7, "trading_days_per_month": 21}},
    {"factor_name": "mfd_sellamt_nd", "feature_name": "mfd_sellamt_nd_1d", "params": {"min_cs_count": 30, "min_observations": None, "n_days": 1, "neutralize_industry": True, "winsor_k": 3.0}},
    {"factor_name": "hml_r_std_nm", "feature_name": "hml_r_std_nm_5m", "params": {"min_cs_count": 100, "min_ts_observations": 80, "n_months": 5, "neutralize_industry": True, "trading_days_per_month": 21, "winsor_lower": 0.01, "winsor_upper": 0.99}},
    {"factor_name": "bias_std_turn_nd", "feature_name": "bias_std_turn_nd_5d", "params": {"long_window": 504, "min_cs_count": 80, "min_long_observations": 2, "min_short_observations": 2, "neutralize_industry": True, "short_window": 5, "winsor_k": 5.0}},
    {"factor_name": "turnover_mean_nm", "feature_name": "turnover_mean_nm_1m", "params": {"n_months": 1, "trading_days_per_month": 21}},
    {"factor_name": "exp_wgt_return_nm", "feature_name": "exp_wgt_return_nm_6m", "params": {"n_months": 6, "trading_days_per_month": 21}},
    {"factor_name": "return_std_nd", "feature_name": "return_std_nd_20d", "params": {"window": 20}},
    {"factor_name": "turnover_bias_nm", "feature_name": "turnover_bias_nm_12m", "params": {"n_months": 12, "trading_days_per_month": 21}},
    {"factor_name": "ff3_residual_volatility_nm", "feature_name": "ff3_residual_volatility_nm_36m", "params": {"market_index": "csi_all_share", "n_months": 36, "trading_days_per_month": 21}},
    {"factor_name": "book_to_price", "feature_name": "book_to_price", "params": {"min_cs_count": 30, "neutralize_industry": True, "winsor_k": 5.0}},
]


def _find_project_root():
    hint = Path(globals().get("PROJECT_ROOT", "/home/aiuser/work")).expanduser()
    candidates = [hint, *hint.parents, Path.cwd(), *Path.cwd().parents]
    return next((path.resolve() for path in candidates if (path / "factor_lib").is_dir()), None)


PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("未找到项目根目录；BigQuant 中通常为 /home/aiuser/work。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SVM_MODULE_PATH = (
    PROJECT_ROOT / "factor_lib" / "Factor Repository"
    / "machine_learning_factors" / "svm_score.py"
)
MODEL_PACKAGE_PARENT_DIR = (
    PROJECT_ROOT / "factor_lib" / "model_artifacts" / "svm_model_bundles"
)
if not SVM_MODULE_PATH.is_file():
    raise FileNotFoundError(f"找不到 SVM 因子脚本：{SVM_MODULE_PATH}")

module_spec = importlib.util.spec_from_file_location("svm_score_training", SVM_MODULE_PATH)
if module_spec is None or module_spec.loader is None:
    raise ImportError(f"无法加载：{SVM_MODULE_PATH}")
svm_module = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = svm_module
module_spec.loader.exec_module(svm_module)

print("[SVM 训练] 开始自动计算 10 个基础因子并训练固定模型。", flush=True)
model_bundle = svm_module.train_svm_model(
    anchor_date=SVM_TRAINING_CONFIG["anchor_date"],
    feature_spec=FEATURE_SPEC,
    training_start_date=SVM_TRAINING_CONFIG["training_start_date"],
    training_end_date=SVM_TRAINING_CONFIG["training_end_date"],
    validation_start_date=SVM_TRAINING_CONFIG["validation_start_date"],
    validation_end_date=SVM_TRAINING_CONFIG["validation_end_date"],
    universe=SVM_TRAINING_CONFIG["universe"],
    label_config=SVM_TRAINING_CONFIG["label_config"],
    preprocessing_config=SVM_TRAINING_CONFIG["preprocessing_config"],
    model_config=SVM_TRAINING_CONFIG["model_config"],
    signal_interval_trading_days=SVM_TRAINING_CONFIG["signal_interval_trading_days"],
    prediction_label_window_trading_days=SVM_TRAINING_CONFIG["prediction_label_window_trading_days"],
    persist_model_bundle=True,
    model_artifact_parent_dir=MODEL_PACKAGE_PARENT_DIR,
    refit_on_train_and_validation=SVM_TRAINING_CONFIG["refit_on_train_and_validation"],
    show_progress=True,
    progress_every=SVM_TRAINING_CONFIG["progress_every"],
)

validation_results = pd.DataFrame(
    model_bundle["training_metadata"]["hyperparameter_results"]
).sort_values("selected_metric", ascending=False, kind="mergesort")
display(validation_results)
display(pd.DataFrame([{
    "model_version": model_bundle["model_version"],
    "validation_rank_ic": model_bundle["training_metadata"]["validation_rank_ic"],
    "validation_auc": model_bundle["training_metadata"]["validation_auc"],
    "rank_ic_cross_section_count": model_bundle["training_metadata"]["validation_rank_ic_cross_section_count"],
    "model_path": model_bundle["artifact_paths"]["model_path"],
    "metadata_path": model_bundle["artifact_paths"]["metadata_path"],
}]))
print("[SVM 训练] 完成：模型包已保存，历史版本未被覆盖。", flush=True)


[SVM 训练] 开始自动计算 10 个基础因子并训练固定模型。
[SVM] 准备训练日历 1/1（100.00%），耗时 0.9s 信号日 98 个，预热 778 日，股票池 all_a                                                                                                                      
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 907,131 行 | 已耗时 3.5s                                                                                                               
[BigQuant 市场日频适配器] [3/3] 市场日频数据准备完成 | 284 行 | 已耗时 0.1s                                                                                                                             
[BigQuant loader] 2/2（100.00%），数据域合并完成，当前 market_daily，284 行，耗时 4.0s                                                                                                               
[id2_std_nm] [2/2] 回归和中性化 12/12 | 100.0% | 当前：2017-11-28 | 已耗时：9.5s | 预计剩余：0.0s0.0s
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 993,477 行 | 已耗时 2.6s                                                                                             

,kernel,C,class_weight,tol,max_iter,cache_size,gamma,selected_metric,auc,rank_ic,rank_ic_cross_section_count,decision_value_positive_sign
1,rbf,0.1,balanced,0.001,-1,600.0,0.03,0.144305,0.585341,0.144305,37,1
7,rbf,0.3,balanced,0.001,-1,600.0,0.10,0.144080,0.585251,0.144080,37,1
2,rbf,0.1,balanced,0.001,-1,600.0,0.10,0.143832,0.584880,0.143832,37,1
6,rbf,0.3,balanced,0.001,-1,600.0,0.03,0.143219,0.584548,0.143219,37,1
15,rbf,3.0,balanced,0.001,-1,600.0,0.01,0.143129,0.584569,0.143129,37,1
10,rbf,1.0,balanced,0.001,-1,600.0,0.01,0.142950,0.584559,0.142950,37,1
12,rbf,1.0,balanced,0.001,-1,600.0,0.10,0.142583,0.584455,0.142583,37,1
21,rbf,10.0,balanced,0.001,-1,600.0,0.03,0.142408,0.584181,0.142408,37,1
11,rbf,1.0,balanced,0.001,-1,600.0,0.03,0.142254,0.583814,0.142254,37,1
16,rbf,3.0,balanced,0.001,-1,600.0,0.03,0.142101,0.583827,0.142101,37,1


,model_version,validation_rank_ic,validation_auc,rank_ic_cross_section_count,model_path,metadata_path
0,svm_all_a_fixed_20260831_223116_5bfc17b8,0.144305,0.585341,37,/home/aiuser/work/factor_lib/model_artifacts/s...,/home/aiuser/work/factor_lib/model_artifacts/s...


[SVM 训练] 完成：模型包已保存，历史版本未被覆盖。
